# Análise de Portabilidade Numérica | Claro Brasil
**Referência:** `docs/brief_claro_portabilidade_eda.md` | `docs/plano_analise_portabilidade_claro.md`

# 01. Setup e Configuração do Ambiente

In [ ]:
# --- Manipulação, Ingestão e Sistema de Arquivos ---
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
# --- Visualização ---
import matplotlib.pyplot as plt
import seaborn as sns
# --- Warnings ---
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# --- Configurações de Exibição do Pandas ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# --- Estilo Visual do Matplotlib e Seaborn ---
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
%config InlineBackend.figure_format = 'retina'

print("Ambiente configurado e bibliotecas importadas com sucesso!")

# 02. Ingestão, Saneamento e Validação Relacional

### Ingestão de Dados SQL e Carga dos DataFrames

In [ ]:
# caminho do aquivo sql
caminho_sql = Path('..')/'data'/'raw'/'migracao_claro_brasil.sql'

# ler o conteúdo do arquivo como string
with open(caminho_sql, mode='r', encoding='utf-8') as f:
    sql_script = f.read()

# criar conexão SQLite em memória e executar o script
conn = sqlite3.connect(':memory:')
conn.executescript(sql_script)

# carregar cada tabela como DataFrame
df_fato = pd.read_sql('SELECT * FROM FatoMigracao', conn)
df_cliente = pd.read_sql('SELECT * FROM DimCliente', conn)
df_servico = pd.read_sql('SELECT * FROM DimServico', conn)
df_operadora = pd.read_sql('SELECT * FROM DimOperadora', conn)
df_calendario = pd.read_sql('SELECT * FROM dCalendario', conn)

# validação
for nome, df in [('Fato', df_fato), ('Cliente', df_cliente), ('Servico', df_servico),
                ('Operadora', df_operadora), ('Calendario', df_calendario)]:
    print(f'{nome}: {df.shape[0]} linhas x {df.shape[1]} colunas')

### Coerção de Tipos

In [ ]:
# Conversão de colunas de data para datetime
df_fato['id_data'] = pd.to_datetime(df_fato['id_data'])
df_calendario['Data'] = pd.to_datetime(df_calendario['Data'])

# Conversão de colunas categóricas de baixa cardinalidade
cols_categoricas = [
    'direcao', 'categoria_servico', 'status_portabilidade',
    'uf', 'regiao', 'motivo', 'canal',
    'operadora_origem', 'operadora_destino'
]

for col in cols_categoricas:
    df_fato[col] = df_fato[col].astype('category')

### Verificação de Nulos e Duplicatas

In [ ]:
dfs = {
    'Fato': df_fato,
    'Cliente': df_cliente,
    'Servico': df_servico,
    'Operadora': df_operadora,
    'Calendario': df_calendario
}

for nome, df in dfs.items():
    nulos = df.isnull().sum().sum()
    duplicatas = df.duplicated().sum()
    print(f'{nome}: {nulos} nulos | {duplicatas} duplicatas')

**Resultado:** Nenhum nulo ou duplicata encontrado nas 5 tabelas. 
As anomalias injetadas pelo gerador provavelmente se manifestam como 
outliers em variáveis numéricas ou inconsistências categóricas.

### Renomeação de Colunas

In [ ]:
df_fato = df_fato.rename(columns={
    'direcao': 'direcao_migracao',
    'motivo': 'motivo_migracao',
    'canal': 'canal_aquisicao'
})

### Auditoria de Colunas Desnormalizadas

In [ ]:
# Merge: geolocalização da Fato vs. DimCliente
df_audit_geo = df_fato[['id_migracao', 'id_cliente', 'uf', 'regiao']].merge(
    df_cliente[['id_cliente', 'uf', 'regiao']],
    on='id_cliente',
    how='left',
    suffixes=('_fato', '_cliente')
)

# Contagem de divergências
divergencias_uf = (df_audit_geo['uf_fato'] != df_audit_geo['uf_cliente']).sum()
divergencias_regiao = (df_audit_geo['regiao_fato'] != df_audit_geo['regiao_cliente']).sum()
pct_uf = (divergencias_uf / len(df_audit_geo)) * 100
pct_regiao = (divergencias_regiao / len(df_audit_geo)) * 100

print("=== AUDITORIA: GEOLOCALIZAÇÃO (Fato vs. DimCliente) ===")
print(f"Total de registros : {len(df_audit_geo):,}")
print(f"Divergências de UF : {divergencias_uf:,} ({pct_uf:.2f}%)")
print(f"Divergências de Região : {divergencias_regiao:,} ({pct_regiao:.2f}%)")

# Amostra de divergências
df_divergencias = df_audit_geo[
    (df_audit_geo['uf_fato'] != df_audit_geo['uf_cliente']) |
    (df_audit_geo['regiao_fato'] != df_audit_geo['regiao_cliente'])
]

if not df_divergencias.empty:
    print(f"\nAmostra das divergências ({len(df_divergencias)} encontradas):")
    display(df_divergencias.head())
else:
    print("\n[OK] Nenhuma divergência — colunas desnormalizadas validadas.")

**Auditoria geolocalização (Fato vs. DimCliente):** 0 divergências em UF e Região 
nos 50.000 registros. Colunas desnormalizadas na Fato são confiáveis para uso 
direto nas análises. A decisão de não fazer merge obrigatório com DimCliente 
para cortes geográficos está validada.

In [ ]:
# Merge da Fato com a DimServico trazendo as colunas específicas de categoria
df_audit_servico = df_fato[['id_migracao', 'id_servico', 'categoria_servico']].merge(
    df_servico[['id_servico', 'categoria']],
    on='id_servico',
    how='left'
)

# Comparação booleana entre categoria_servico (Fato) e categoria (DimServico)
divergencias_cat = (df_audit_servico['categoria_servico'] != df_audit_servico['categoria']).sum()
pct_cat = (divergencias_cat / len(df_audit_servico)) * 100

print("=== RELATÓRIO DE AUDITORIA: CATEGORIA DE SERVIÇO ===")
print(f"Total de registros na Fato : {len(df_audit_servico):,}")
print(f"Divergências de Categoria : {divergencias_cat:,} ({pct_cat:.2f}%)")

# Filtrar e exibir amostra das divergências encontradas
df_divergencias_cat = df_audit_servico[
    df_audit_servico['categoria_servico'] != df_audit_servico['categoria']
]
if not df_divergencias_cat.empty:
    print("\nAmostra das divergências encontradas (Primeiras 5 linhas):")
    display(df_divergencias_cat.head())
else:
    print("\nNenhuma divergência encontrada: Fato e DimServico estão 100% alinhadas.")

**Auditoria categoria de serviço (Fato vs. DimServico):** 0 divergências nos 
50.000 registros. Ambas as auditorias (geolocalização + categoria) confirmam 
que as colunas desnormalizadas da FatoMigracao são consistentes com as dimensões no
uso direto validado para todas as análises.

# 03. (EDA) e Detecção de Anomalias

### Checagem da Janela Temporal

In [ ]:
# Extração de ano-mês a partir de id_data
df_fato['ano_mes'] = df_fato['id_data'].dt.to_period('M')

# Contagem de registros por mês
cont_registros = df_fato['ano_mes'].value_counts().sort_index()

# Intervalo temporal completo esperado (jan/2022 a dez/2024)
intervalo_esperado = pd.period_range(start='2022-01', end='2024-12', freq='M')

# Reindexação para capturar meses zerados
completude_mensal = cont_registros.reindex(intervalo_esperado, fill_value=0)

# Métricas de integridade
data_min = df_fato['id_data'].min()
data_max = df_fato['id_data'].max()
meses_presentes = (completude_mensal > 0).sum()
meses_zerados = (completude_mensal == 0).sum()

print(f"Janela temporal : {data_min.date()} a {data_max.date()}")
print(f"Meses esperados : {len(intervalo_esperado)}")
print(f"Meses com dados : {meses_presentes}")
print(f"Meses sem dados : {meses_zerados}")
print(f"Registros por mês : média {completude_mensal.mean():.0f} | min {completude_mensal.min()} | max {completude_mensal.max()}")

**Checagem temporal:** Janela completa (jan/2022 a dez/2024), 36 meses sem lacunas.
Volume mensal estável (média 1.389, amplitude ~15%). Sem anomalias de volumetria.

### Distribuição e Outliers em Variáveis Numéricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=df_fato, y='valor_mensal_servico', ax=axes[0])
axes[0].set_title('Valor Mensal do Serviço (R$)')

sns.boxplot(data=df_fato, y='tempo_permanencia_dias', ax=axes[1])
axes[1].set_title('Tempo de Permanência (dias)')

sns.boxplot(data=df_fato, y='nota_satisfacao', ax=axes[2])
axes[2].set_title('Nota de Satisfação')

plt.tight_layout()
plt.show()

In [ ]:
# Describe das variáveis numéricas de interesse
df_fato[['valor_mensal_servico', 'tempo_permanencia_dias', 'nota_satisfacao']].describe()

### Detecção de Concept Drift

In [ ]:
# Concept drift: valor_mensal_servico por ano
df_fato['ano'] = df_fato['id_data'].dt.year
df_fato.groupby('ano')['valor_mensal_servico'].describe()

In [ ]:
# Concept drift: proporção IN vs OUT por ano
df_fato.groupby('ano')['direcao_migracao'].value_counts(normalize=True).unstack()

### Inspeção de Variáveis Categóricas

In [ ]:
# --- 03.4 Inspeção de Variáveis Categóricas ---
cols_inspecao = [
    'direcao_migracao', 'categoria_servico', 'status_portabilidade',
    'motivo_migracao', 'canal_aquisicao', 'operadora_origem', 'operadora_destino'
]

for col in cols_inspecao:
    print(f'--- {col} ---')
    print(df_fato[col].value_counts())
    print()

**Achado relevante:** identificado 4º status de portabilidade não previsto 
no briefing — "Rejeitada pela Anatel" (2.041 registros, 4,08%). 
Provável anomalia injetada pelo gerador. Decisão: tratar separadamente 
na análise, não incluir no cálculo de Net Migration.

Demais categóricas sem inconsistências: sem typos, sem categorias 
inesperadas, distribuições equilibradas entre os valores.

### Resumo

**Janela temporal:** 36 meses completos (jan/2022 a dez/2024), sem lacunas.
Volume mensal estável (média 1.389, amplitude ~15%).

**Variáveis numéricas:**
- `valor_mensal_servico`: poucos outliers no topo (~R$411), consistentes com planos premium. Sem anomalias.
- `tempo_permanencia_dias`: distribuição simétrica (atípico para tenure real, artefato do gerador). Sem outliers.
- `nota_satisfacao`: distribuição uniforme de 1 a 10, dentro da escala esperada.

**Concept drift:** não detectado. Pricing estável nos 3 anos.
Proporção IN/OUT constante em 52/48. Possíveis drifts mais sutis
serão investigados nos cortes por categoria e região.

**Variáveis categóricas:** identificado 4º status de portabilidade não
previsto no briefing como, "Rejeitada pela Anatel" (2.041 registros, 4,08%).
Decisão: tratar separadamente, não incluir no cálculo de Net Migration.
Demais categóricas sem inconsistências.

**Decisão:** dados aptos para análise.